# BUS 104: Finsimco Portfolio Optimization Starter Notebook

Welcome! This notebook scales our 3-stock lecture example up to **25 global companies** for your **Finsimco Portfolio Management Simulation**.

### How to Use This Notebook (3 Easy Steps)
1. **Run cells from top to bottom:** Click the **Play button (▶)** on each code cell in order (or press `Shift + Enter`).
2. **Upload your Excel file** from Finsimco (`History` tab $\to$ **Download year historical data**) when Step 1 asks for it.
3. **Choose your team's mandate** in Step 4:
   - **Odd Teams (1, 3, 5, ...): Hedge Fund** $\to$ Maximize Sharpe Ratio.
   - **Even Teams (2, 4, 6, ...): Pension Fund** $\to$ Maximize Sharpe Ratio **subject to a volatility cap** so your risk stays below the Pension Fund median.


## Step 1: Import Libraries & Upload Your Finsimco Historical Data


In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize, LinearConstraint, NonlinearConstraint

# Upload the Excel file downloaded from Finsimco (works in Google Colab)
try:
    from google.colab import files
    print("Click 'Choose Files' below and select your Finsimco Excel file (.xlsx)...")
    uploaded = files.upload()
    filename = list(uploaded.keys())[0]
    print(f"Uploaded: {filename}")
except ImportError:
    # Fallback if running locally on your computer
    filename = "historical_data.xlsx"
    print(f"Running locally — looking for: {filename}")


## Step 2: Load & Clean the Price Data
Run this cell to read the Excel file and inspect the first 5 rows. It automatically keeps only the numeric stock price columns.


In [ ]:
# Read the Excel file (assumes first column is Date; numeric columns are stock prices)
raw = pd.read_excel(filename, index_col=0)

# Keep only numeric price columns and drop any empty rows
prices = raw.select_dtypes(include=[np.number]).dropna(how="all")

print(f"Loaded {prices.shape[0]} trading days across {prices.shape[1]} stocks.")
prices.head()


## Step 3: Compute Expected Returns ($\mu$) and Covariance Matrix ($\Sigma$)
We compute daily percentage returns and annualize them using **252 trading days per year**.


In [ ]:
# Daily percentage returns
returns = prices.pct_change().dropna()

# Annualized expected returns (mu) and 25x25 covariance matrix (Sigma)
mu = returns.mean().values * 252
Sigma = returns.cov().values * 252
tickers = list(prices.columns)
n = len(tickers)

# Risk-free rate (you can adjust if needed; 3% = 0.03)
rf = 0.03

def portfolio_stats(w):
    """Return (expected_return, volatility, sharpe_ratio) for weights w."""
    port_ret = float(np.dot(mu, w))
    port_vol = float(np.sqrt(np.dot(w, np.dot(Sigma, w))))
    sharpe = (port_ret - rf) / port_vol if port_vol > 0 else 0.0
    return port_ret, port_vol, sharpe

print(f"Ready to optimize across {n} stocks!")


## Step 4A: Hedge Fund Optimization (Odd Teams: 1, 3, 5, ...)
Goal: **Maximize Sharpe Ratio** (which in `scipy.optimize.minimize` means minimizing the **negative** Sharpe Ratio).

> *Optional setting:* `MAX_WEIGHT = 1.0` allows up to 100% in one stock; setting `MAX_WEIGHT = 0.25` forces diversification (no more than 25% in any single stock).


In [ ]:
MAX_WEIGHT = 0.25   # Change to 1.0 if you want unconstrained weights, or e.g. 0.20 for max 20% per stock
CAPITAL = 100_000_000  # Total portfolio capital in $ (check your Finsimco Trading tab!)

def neg_sharpe(w):
    ret, vol, sharpe = portfolio_stats(w)
    return -sharpe

budget_con = LinearConstraint(np.ones(n), 1.0, 1.0)
bounds = [(0.0, MAX_WEIGHT)] * n
x0 = np.ones(n) / n

res_hf = minimize(neg_sharpe, x0=x0, method="SLSQP", bounds=bounds, constraints=[budget_con])

w_hf = res_hf.x
ret_hf, vol_hf, sharpe_hf = portfolio_stats(w_hf)

print("=== HEDGE FUND OPTIMAL PORTFOLIO ===")
print(f"Expected Annual Return : {ret_hf:.2%}")
print(f"Annual Volatility      : {vol_hf:.2%}")
print(f"Sharpe Ratio           : {sharpe_hf:.3f}")

df_hf = pd.DataFrame({
    "Stock": tickers,
    "Weight (%)": np.round(w_hf * 100, 2),
    "Dollar Allocation ($)": np.round(w_hf * CAPITAL, 0)
}).sort_values("Weight (%)", ascending=False)

df_hf[df_hf["Weight (%)"] > 0.1].reset_index(drop=True)


## Step 4B: Pension Fund Optimization (Even Teams: 2, 4, 6, ...)
Goal: **Maximize Sharpe Ratio** while keeping **portfolio volatility ($\sigma_p$) below the Pension Fund median**.

Adjust `VOL_CAP` below (for example, `0.16` for 16% annual volatility) to control your risk!


In [ ]:
VOL_CAP = 0.16      # Maximum allowed annual volatility (e.g., 0.16 = 16%)
MAX_WEIGHT_PF = 0.20

def port_vol_fn(w):
    return np.sqrt(np.dot(w, np.dot(Sigma, w)))

vol_con = NonlinearConstraint(port_vol_fn, 0.0, VOL_CAP)
bounds_pf = [(0.0, MAX_WEIGHT_PF)] * n

res_pf = minimize(
    neg_sharpe,
    x0=np.ones(n) / n,
    method="SLSQP",
    bounds=bounds_pf,
    constraints=[budget_con, vol_con]
)

if res_pf.success:
    w_pf = res_pf.x
    ret_pf, vol_pf, sharpe_pf = portfolio_stats(w_pf)
    print("=== PENSION FUND OPTIMAL PORTFOLIO ===")
    print(f"Expected Annual Return : {ret_pf:.2%}")
    print(f"Annual Volatility      : {vol_pf:.2%}  (Cap: {VOL_CAP:.2%})")
    print(f"Sharpe Ratio           : {sharpe_pf:.3f}")

    df_pf = pd.DataFrame({
        "Stock": tickers,
        "Weight (%)": np.round(w_pf * 100, 2),
        "Dollar Allocation ($)": np.round(w_pf * CAPITAL, 0)
    }).sort_values("Weight (%)", ascending=False)
    display_df = df_pf[df_pf["Weight (%)"] > 0.1].reset_index(drop=True)
else:
    print("Optimization did not converge — try slightly increasing VOL_CAP (e.g. 0.18 or 0.20).")
    display_df = pd.DataFrame()

display_df


## Step 5: Save Your Notebook for Canvas Submission
1. In Google Colab, click **File $\to$ Download $\to$ Download `.ipynb`**.
2. Upload that `.ipynb` file (along with your Finsimco Certificate and 2–4 sentence summary) to the **Finsimco Case Study: Individual Submission** quiz on Canvas by **Friday, Dec 4 at 11:59 PM**.
